# Train and validate the sex classifier

Research only: fine-tune an ImageNet-pretrained ResNet-18 on human-labelled crops
and validate it by holding out one recording date at a time. Human labels provide
the training targets and evaluation reference; each prediction uses its own crop.
No expected male count is supplied to the model.

Run from `notebooks/` with `../data/labeled_crops.npz`, or upload that file into the
working directory on Colab. Use only trusted dataset files: crop provenance is
stored as a NumPy object array and requires `allow_pickle=True`.

The later cells train six validation models and two deployment models with the
default settings. A GPU is recommended. The first model construction downloads
ImageNet weights unless they are already cached. Routine scoring uses saved model
weights and does not require this notebook. Deployment weights are saved in
`../model/`; copy them deliberately if replacing a released model.


## Imports and configuration

In [ ]:
# Import dependencies
import collections
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from torchvision import models

In [ ]:
# Find a local dataset or a Colab upload
def find_data_file(name):
    for candidate in (Path("../data") / name, Path(name)):
        if candidate.exists():
            return candidate
    raise ValueError(f"Missing {name}; put it in ../data or the current directory")


# Define file locations
DATASET_PATH = find_data_file(name="labeled_crops.npz")
MODEL_DIRECTORY = Path("../model")

# Define training hyperparameters
EPOCHS = 40
BATCH_SIZE = 64
LEARNING_RATE = 6e-4
WEIGHT_DECAY = 1e-2
FROZEN_LAYERS = 6
SEEDS = (0, 1)
TEST_TIME_AUGMENTATIONS = 6

# Call a fly male above this score, and call the score confident outside the band
DECISION_THRESHOLD = 0.5
CONFIDENT_LOW = 0.1
CONFIDENT_HIGH = 0.9

# Define augmentation strength
ROTATION_LIMIT = 8 * np.pi / 180
TRANSLATION_LIMIT = 0.12
SCALE_RANGE = (0.92, 1.08)
BRIGHTNESS_RANGE = (0.85, 1.15)
BRIGHTNESS_SHIFT = 0.14

In [ ]:
# Select the compute device
def select_device():
    # Prefer CUDA, then Apple Silicon, then CPU
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


device = select_device()
device_name = torch.cuda.get_device_name(0) if device.type == "cuda" else device.type
print(f"Device: {device_name}")

## Function definitions

In [ ]:
# Build a single-channel ResNet-18 from ImageNet weights
def build_model(frozen_layers):
    # Load the pretrained network
    network = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

    # Collapse the three-channel stem into one channel
    summed_weights = network.conv1.weight.data.sum(dim=1, keepdim=True)
    network.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    network.conv1.weight.data = summed_weights

    # Freeze the early layers
    for layer in list(network.children())[:frozen_layers]:
        for parameter in layer.parameters():
            parameter.requires_grad = False

    # Replace the classifier head
    network.fc = nn.Linear(512, 1)

    # Return the adapted network
    return network

In [ ]:
# Apply random augmentations to a batch
def augment_batch(images):
    # Flip vertically, since the body axis is horizontal
    if torch.rand(1) < 0.5:
        images = torch.flip(images, dims=[2])

    # Flip horizontally, since the head-tail call is unreliable
    if torch.rand(1) < 0.5:
        images = torch.flip(images, dims=[3])

    # Sample per-image affine parameters
    batch_size = len(images)
    angles = (torch.rand(batch_size, device=images.device) - 0.5) * 2 * ROTATION_LIMIT
    scales = SCALE_RANGE[0] + (SCALE_RANGE[1] - SCALE_RANGE[0]) * torch.rand(
        batch_size, device=images.device
    )
    shift_x = (torch.rand(batch_size, device=images.device) - 0.5) * TRANSLATION_LIMIT
    shift_y = (torch.rand(batch_size, device=images.device) - 0.5) * TRANSLATION_LIMIT

    # Assemble the affine matrices
    cosine = torch.cos(angles) / scales
    sine = torch.sin(angles) / scales
    theta = torch.stack(
        [
            torch.stack([cosine, -sine, shift_x], dim=1),
            torch.stack([sine, cosine, shift_y], dim=1),
        ],
        dim=1,
    )

    # Resample the batch
    grid = nn.functional.affine_grid(theta, images.shape, align_corners=False)
    images = nn.functional.grid_sample(
        images,
        grid,
        align_corners=False,
        padding_mode="border",
    )

    # Jitter brightness
    gain = BRIGHTNESS_RANGE[0] + (
        BRIGHTNESS_RANGE[1] - BRIGHTNESS_RANGE[0]
    ) * torch.rand(batch_size, 1, 1, 1, device=images.device)
    offset = BRIGHTNESS_SHIFT * (
        torch.rand(batch_size, 1, 1, 1, device=images.device) - 0.5
    )

    # Return the augmented batch
    return images * gain + offset

In [ ]:
# Train one model on the given crops
def train_model(train_images, train_labels, seed):
    # Seed the run
    torch.manual_seed(seed)

    # Build the network and optimiser
    network = build_model(frozen_layers=FROZEN_LAYERS).to(device)
    trainable = [parameter for parameter in network.parameters() if parameter.requires_grad]
    optimizer = torch.optim.AdamW(
        trainable,
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    # Weight the loss by the class imbalance
    positive_weight = torch.tensor(
        [float((train_labels == 0).sum()) / float((train_labels == 1).sum())],
        dtype=torch.float32,
        device=device,
    )
    criterion = nn.BCEWithLogitsLoss(pos_weight=positive_weight)

    # Move the data onto the device
    image_tensor = torch.tensor(train_images, device=device).unsqueeze(1)
    label_tensor = torch.tensor(train_labels, device=device)

    # Schedule the learning rate over the whole run
    steps_per_epoch = max(1, len(image_tensor) // BATCH_SIZE)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=LEARNING_RATE,
        total_steps=EPOCHS * steps_per_epoch,
    )

    # Run the training epochs
    for epoch in range(EPOCHS):
        network.train()
        order = torch.randperm(len(image_tensor), device=device)
        for start in range(0, len(order) - BATCH_SIZE + 1, BATCH_SIZE):
            batch_indices = order[start : start + BATCH_SIZE]
            optimizer.zero_grad(set_to_none=True)
            logits = network(augment_batch(image_tensor[batch_indices])).squeeze(1)
            loss = criterion(logits, label_tensor[batch_indices])
            loss.backward()
            optimizer.step()
            scheduler.step()

    # Return the trained network
    return network

In [ ]:
# Average one network's predictions over several augmentations
def predict_with_augmentation(network, test_images):
    # Move the crops onto the device
    test_tensor = torch.tensor(test_images, device=device).unsqueeze(1)

    # Score the clean crops, then several augmented copies
    network.eval()
    predictions = []
    with torch.no_grad():
        clean = torch.sigmoid(network(test_tensor).squeeze(1))
        predictions.append(clean.cpu().numpy())

        for _ in range(TEST_TIME_AUGMENTATIONS - 1):
            scores = torch.sigmoid(network(augment_batch(test_tensor)).squeeze(1))
            predictions.append(scores.cpu().numpy())

    # Return the mean probability per fly
    return np.mean(predictions, axis=0)

In [ ]:
# Average predictions over several seeds
def fit_ensemble(train_images, train_labels, test_images, seeds):
    # Train one model per seed and score the held-out crops with each
    predictions = []
    for seed in seeds:
        network = train_model(
            train_images=train_images,
            train_labels=train_labels,
            seed=seed,
        )
        predictions.append(predict_with_augmentation(network, test_images))

    # Return the mean probability
    return np.mean(predictions, axis=0)

## Load the dataset

In [ ]:
# Load the crops and their provenance
dataset = np.load(DATASET_PATH, allow_pickle=True)
crops = dataset["crops"]
labels = dataset["y"].astype(np.float32)
sources = dataset["meta"]

# Validate crop, label, and provenance alignment
if crops.ndim != 3 or crops.shape[1:] != (96, 96):
    raise ValueError(f"Expected crops with shape (n, 96, 96), got {crops.shape}")
if labels.shape != (len(crops),) or sources.shape != (len(crops), 4):
    raise ValueError("Crop, label, and provenance lengths must match")
if not np.isin(labels, [0, 1]).all():
    raise ValueError("Training labels must contain only 0 and 1")
if min(int((labels == 0).sum()), int((labels == 1).sum())) < 8:
    raise ValueError("At least eight labelled crops per class are required for this notebook")

# Derive the grouping columns
wells = np.asarray([row[0] for row in sources])
frames = np.asarray([row[2] for row in sources])
dates = np.asarray([well[:4] for well in wells])

# Normalise the images for the network
images = (crops.astype(np.float32) / 255.0 - 0.5) / 0.25

print(f"Dataset: {len(labels)} crops | {int(labels.sum())} male | {int((1 - labels).sum())} female")
print()
for date in sorted(set(dates)):
    selected = dates == date
    well_list = ", ".join(sorted(set(wells[selected])))
    print(f"  {date} | {selected.sum():4d} crops | {well_list}")

## Sanity check the crops

In [ ]:
# Show a sample of each class
figure, axes = plt.subplots(2, 8, figsize=(14, 3.8))
generator = np.random.default_rng(0)

for row, (label_value, label_name) in enumerate([(1, "male"), (0, "female")]):
    sample = generator.choice(np.where(labels == label_value)[0], 8, replace=False)
    for column, index in enumerate(sample):
        axes[row, column].imshow(crops[index], cmap="gray")
        axes[row, column].axis("off")
    axes[row, 0].set_title(label_name, loc="left", fontsize=11)

plt.tight_layout()
plt.show()

## Validate by holding out one recording date

Consecutive frames of the same fly are near-duplicates, so validation holds out
whole recording dates. This estimates performance across the dates represented
in the dataset; it does not establish accuracy on every future recording.

A binary evaluation call uses a score above 0.5. The confident share is the
fraction of crops outside the 0.1 to 0.9 score band. Confidence is a model score
heuristic; inspect both its coverage and held-out accuracy before choosing a
threshold. Routine inference exposes each fly's score and rank without a count quota.


In [ ]:
# Hold out each date in turn
fold_results = []
held_out_probabilities = np.full(len(labels), np.nan, dtype=np.float32)
held_out_dates = sorted(set(dates))

for held_out in held_out_dates:
    # Split by recording date
    test_selector = dates == held_out
    train_selector = ~test_selector

    # Train an ensemble and predict the held-out date
    probabilities = fit_ensemble(
        train_images=images[train_selector],
        train_labels=labels[train_selector],
        test_images=images[test_selector],
        seeds=SEEDS,
    )
    held_out_probabilities[test_selector] = probabilities

    # Call every fly on its own score
    calls = (probabilities > DECISION_THRESHOLD).astype(int)
    truth = labels[test_selector]

    # Score the fold, and score it again on the confident calls alone
    confident = (probabilities < CONFIDENT_LOW) | (probabilities > CONFIDENT_HIGH)
    fold_results.append(
        {
            "held_out": held_out,
            "crops": int(test_selector.sum()),
            "auc": roc_auc_score(truth, probabilities),
            "accuracy": float((calls == truth).mean()),
            "confident_share": float(confident.mean()),
            "confident_accuracy": float((calls[confident] == truth[confident]).mean()),
        }
    )

    print(
        f"Fold {len(fold_results)} | "
        f"Held out 2026-{held_out[:2]}-{held_out[2:]} | "
        f"AUC: {fold_results[-1]['auc']:.3f} | "
        f"Accuracy: {fold_results[-1]['accuracy']:.3f} | "
        f"Confident {fold_results[-1]['confident_share']:.0%} of crops at {fold_results[-1]['confident_accuracy']:.3f}"
    )

In [ ]:
# Report the mean across folds
for name in ("auc", "accuracy", "confident_share", "confident_accuracy"):
    print(f"Mean {name.replace('_', ' ')}: {np.mean([result[name] for result in fold_results]):.3f}")

# Save the held-out predictions next to the dataset
np.savez_compressed(
    DATASET_PATH.with_name("cv_predictions.npz"),
    meta=sources,
    labels=labels,
    probabilities=held_out_probabilities,
    calls=(held_out_probabilities > DECISION_THRESHOLD).astype(int),
    dates=dates,
)
print(f"Saved {DATASET_PATH.with_name('cv_predictions.npz')}")

## Train the deployment model on every labelled fly

In [ ]:
# Train and save one model per seed
MODEL_DIRECTORY.mkdir(parents=True, exist_ok=True)

for seed in SEEDS:
    network = train_model(train_images=images, train_labels=labels, seed=seed)
    checkpoint_path = MODEL_DIRECTORY / f"flysex_seed{seed}.pt"
    torch.save(network.state_dict(), checkpoint_path)
    print(f"Saved {checkpoint_path}")